In [1]:

import json
import pickle
import time
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

start = time.time()

# ----------------------------
# 0) Project / MLflow setup
# ----------------------------
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data"
PREP_ROOT = DATA_ROOT / "prepared"
REPORT_ROOT = PROJECT_ROOT / "reports"
ARTIFACT_ROOT = PROJECT_ROOT / "models" / "artifacts"
MLRUNS_ROOT = PROJECT_ROOT / "mlruns"

REPORT_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
MLRUNS_ROOT.mkdir(parents=True, exist_ok=True)

tracking_uri = f"file:///{MLRUNS_ROOT.resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("recomart_item_based_cf")

FILE_PATH = PREP_ROOT / "interactions_prepared.csv"

# ----------------------------
# 1) Load prepared interactions
# ----------------------------
use_cols = ["user_id", "item_id", "event_weight"]
df = pd.read_csv(FILE_PATH, usecols=use_cols)

required_cols = {"user_id", "item_id", "event_weight"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Loaded:", FILE_PATH)
print("Raw shape:", df.shape)

# ----------------------------
# 2) Aggregate + reduce dataset
# ----------------------------
df = (
    df.groupby(["user_id", "item_id"], as_index=False, sort=False)["event_weight"]
      .sum()
)

MIN_USER_ITEMS = 5
MIN_ITEM_USERS = 20
MAX_USERS = 30000
MAX_ITEMS = 3000
TOP_NEIGHBORS = 100
MAX_EVAL_USERS = 10000

changed = True
while changed:
    before = len(df)

    user_counts = df.groupby("user_id")["item_id"].nunique()
    keep_users = user_counts[user_counts >= MIN_USER_ITEMS].index
    df = df[df["user_id"].isin(keep_users)]

    item_counts = df.groupby("item_id")["user_id"].nunique()
    keep_items = item_counts[item_counts >= MIN_ITEM_USERS].index
    df = df[df["item_id"].isin(keep_items)]

    changed = len(df) != before

top_users = (
    df.groupby("user_id")["item_id"]
      .nunique()
      .sort_values(ascending=False)
      .head(MAX_USERS)
      .index
)
df = df[df["user_id"].isin(top_users)]

top_items = (
    df.groupby("item_id")["user_id"]
      .nunique()
      .sort_values(ascending=False)
      .head(MAX_ITEMS)
      .index
)
df = df[df["item_id"].isin(top_items)].copy()

if df.empty:
    raise ValueError("Filtered dataset is empty. Relax thresholds.")

print("Filtered shape:", df.shape)
print("Users:", df["user_id"].nunique(), "Items:", df["item_id"].nunique())

# ----------------------------
# 3) Leave-one-out split
# ----------------------------
rng = np.random.RandomState(42)
df["_rand"] = rng.rand(len(df))
test_idx = df.groupby("user_id")["_rand"].idxmin()

test_df = df.loc[test_idx, ["user_id", "item_id", "event_weight"]].copy()
train_df = df.drop(index=test_idx).copy()

train_counts = train_df.groupby("user_id")["item_id"].size()
valid_users = train_counts[train_counts >= 1].index
train_df = train_df[train_df["user_id"].isin(valid_users)].copy()
test_df = test_df[test_df["user_id"].isin(valid_users)].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# ----------------------------
# 4) Build sparse matrix
# ----------------------------
user_ids = train_df["user_id"].unique()
item_ids = train_df["item_id"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}
idx_to_item = np.array(item_ids)

test_df = test_df[
    test_df["user_id"].isin(user_to_idx) & test_df["item_id"].isin(item_to_idx)
].copy()

rows = train_df["user_id"].map(user_to_idx).to_numpy()
cols = train_df["item_id"].map(item_to_idx).to_numpy()
vals = train_df["event_weight"].astype(np.float32).to_numpy()

user_item = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(user_to_idx), len(item_to_idx)),
    dtype=np.float32
).tocsr()

print("User-item matrix:", user_item.shape)

# ----------------------------
# 5) Train fast item neighbors
# ----------------------------
item_user = user_item.T.tocsr()

nn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=min(TOP_NEIGHBORS + 1, item_user.shape[0]),
    n_jobs=-1
)
nn.fit(item_user)

distances, neighbors = nn.kneighbors(item_user, return_distance=True)
sims = 1.0 - distances

print("Neighbor index shape:", neighbors.shape)

# ----------------------------
# 6) Cache seen items once
# ----------------------------
user_seen_idx = {}
user_seen_vals = {}

for user_id, uidx in user_to_idx.items():
    start_ptr = user_item.indptr[uidx]
    end_ptr = user_item.indptr[uidx + 1]
    seen_idx = user_item.indices[start_ptr:end_ptr]
    seen_vals = user_item.data[start_ptr:end_ptr]
    user_seen_idx[user_id] = seen_idx
    user_seen_vals[user_id] = seen_vals

# ----------------------------
# 7) Recommend
# ----------------------------
def recommend_items(user_id, k=10):
    if user_id not in user_to_idx:
        return []

    seen_idx = user_seen_idx[user_id]
    seen_vals = user_seen_vals[user_id]

    scores = np.zeros(len(item_to_idx), dtype=np.float32)

    for item_idx, weight in zip(seen_idx, seen_vals):
        nbrs = neighbors[item_idx]
        nbr_sims = sims[item_idx]

        mask = nbrs != item_idx
        nbrs = nbrs[mask]
        nbr_sims = nbr_sims[mask]

        scores[nbrs] += weight * nbr_sims

    scores[seen_idx] = -np.inf

    top_n = min(k, len(scores))
    top_idx = np.argpartition(scores, -top_n)[-top_n:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    top_idx = top_idx[np.isfinite(scores[top_idx])]

    return idx_to_item[top_idx].tolist()

sample_user = train_df["user_id"].iloc[0]
sample_recs = recommend_items(sample_user, k=5)
print("Sample user:", sample_user)
print("Top-5 recs:", sample_recs)

# ----------------------------
# 8) Evaluate
# ----------------------------
def precision_recall_ndcg_at_k(test_df, k=10, max_eval_users=10000):
    user_truth = test_df.groupby("user_id")["item_id"].apply(set).to_dict()
    eval_users = list(user_truth.keys())[:max_eval_users]

    precisions, recalls, ndcgs = [], [], []

    for user_id in eval_users:
        recs = recommend_items(user_id, k=k)
        if not recs:
            continue

        true_items = user_truth[user_id]
        hits = np.array([1 if item in true_items else 0 for item in recs], dtype=np.float32)

        precision = float(hits.sum() / k)
        recall = float(hits.sum() / len(true_items))

        discounts = 1.0 / np.log2(np.arange(2, len(hits) + 2))
        dcg = float((hits * discounts).sum())

        ideal_len = min(len(true_items), k)
        idcg = float((np.ones(ideal_len) / np.log2(np.arange(2, ideal_len + 2))).sum())
        ndcg = dcg / idcg if idcg > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    return {
        f"precision_at_{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"recall_at_{k}": float(np.mean(recalls)) if recalls else 0.0,
        f"ndcg_at_{k}": float(np.mean(ndcgs)) if ndcgs else 0.0,
        f"evaluated_users_at_{k}": int(len(precisions)),
    }

metrics_5 = precision_recall_ndcg_at_k(test_df, k=5, max_eval_users=MAX_EVAL_USERS)
metrics_10 = precision_recall_ndcg_at_k(test_df, k=10, max_eval_users=MAX_EVAL_USERS)

runtime_minutes = (time.time() - start) / 60.0

all_metrics = {
    **metrics_5,
    **metrics_10,
    "runtime_minutes": float(runtime_minutes),
}

print("\nMetrics")
for metric, value in all_metrics.items():
    print(f"{metric}: {value:.4f}" if isinstance(value, float) else f"{metric}: {value}")

# ----------------------------
# 9) Save artifacts
# ----------------------------
metrics_path = REPORT_ROOT / "item_cf_metrics.json"
summary_path = REPORT_ROOT / "item_cf_run_summary.json"
bundle_path = ARTIFACT_ROOT / "item_cf_bundle.pkl"

run_summary = {
    "input_file": str(FILE_PATH),
    "train_shape": list(train_df.shape),
    "test_shape": list(test_df.shape),
    "user_item_shape": list(user_item.shape),
    "sample_user": int(sample_user),
    "sample_top5_recommendations": [int(x) for x in sample_recs],
    "metrics": all_metrics,
}

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2)

with open(bundle_path, "wb") as f:
    pickle.dump(
        {
            "nn_model": nn,
            "user_to_idx": user_to_idx,
            "item_to_idx": item_to_idx,
            "idx_to_item": idx_to_item,
            "neighbors": neighbors,
            "sims": sims,
            "user_seen_idx": user_seen_idx,
            "user_seen_vals": user_seen_vals,
        },
        f,
    )

# ----------------------------
# 10) MLflow logging
# ----------------------------
with mlflow.start_run(run_name="item_based_cf_baseline") as run:
    mlflow.log_param("model_type", "item_based_collaborative_filtering")
    mlflow.log_param("algorithm", "item_knn_cosine")
    mlflow.log_param("input_file", str(FILE_PATH))
    mlflow.log_param("split_type", "random_leave_one_out_after_aggregation")
    mlflow.log_param("min_user_items", MIN_USER_ITEMS)
    mlflow.log_param("min_item_users", MIN_ITEM_USERS)
    mlflow.log_param("max_users", MAX_USERS)
    mlflow.log_param("max_items", MAX_ITEMS)
    mlflow.log_param("top_neighbors", TOP_NEIGHBORS)
    mlflow.log_param("max_eval_users", MAX_EVAL_USERS)

    mlflow.log_param("train_rows", int(len(train_df)))
    mlflow.log_param("test_rows", int(len(test_df)))
    mlflow.log_param("train_users", int(train_df["user_id"].nunique()))
    mlflow.log_param("train_items", int(train_df["item_id"].nunique()))

    mlflow.log_metrics(all_metrics)

    mlflow.log_artifact(str(metrics_path))
    mlflow.log_artifact(str(summary_path))
    mlflow.log_artifact(str(bundle_path))

    mlflow.sklearn.log_model(nn, artifact_path="item_knn_model")

    run_id = run.info.run_id

print("\nMLflow tracking URI:", tracking_uri)
print("MLflow run_id:", run_id)
print("Saved artifacts:")
print("-", metrics_path)
print("-", summary_path)
print("-", bundle_path)


C:\Users\barath\AppData\Local\anaconda3\envs\dm4ml\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Loaded: C:\Users\barath\recomart-pipeline\data\prepared\interactions_prepared.csv
Raw shape: (2755641, 3)


Filtered shape: (30823, 3)
Users: 3204 Items: 777
Train shape: (27619, 4)
Test shape: (3204, 3)
User-item matrix: (3204, 777)
Neighbor index shape: (777, 101)
Sample user: 761633
Top-5 recs: [47526, 235559, 278272, 29825, 465522]



Metrics
precision_at_5: 0.0287
recall_at_5: 0.1433
ndcg_at_5: 0.0939
evaluated_users_at_5: 3204
precision_at_10: 0.0210
recall_at_10: 0.2104
ndcg_at_10: 0.1153
evaluated_users_at_10: 3204
runtime_minutes: 0.1005


2026/04/30 23:12:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/30 23:12:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/04/30 23:12:38 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!



MLflow tracking URI: file:///C:/Users/barath/recomart-pipeline/mlruns
MLflow run_id: 4d770f2096dd4b8eb00726755b149ee8
Saved artifacts:
- C:\Users\barath\recomart-pipeline\reports\item_cf_metrics.json
- C:\Users\barath\recomart-pipeline\reports\item_cf_run_summary.json
- C:\Users\barath\recomart-pipeline\models\artifacts\item_cf_bundle.pkl
